# Integración de los datos

In [10]:
import sys, os, duckdb
import pandas as pd
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Rutas coherentes
features_dir = "../../data/features/"
db_path = "../../db/olist_analytics.duckdb"
views_dir = "../../data/views/"
os.makedirs(os.path.dirname(db_path), exist_ok=True)
os.makedirs(views_dir, exist_ok=True)

con = duckdb.connect(db_path, read_only=False)
con.execute("CREATE SCHEMA IF NOT EXISTS olist")

#  Cargar features y clean archivos
orders  = pd.read_csv(features_dir + "features_orders.csv", parse_dates=[
    "order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
    "order_delivered_customer_date","order_estimated_delivery_date"
])
items   = pd.read_csv(features_dir + "features_items_agg.csv")
reviews = pd.read_csv(features_dir + "features_reviews.csv", parse_dates=[
    "review_creation_date","review_answer_timestamp"
])
order_items = pd.read_csv("../../data/processed/olist_order_items_clean.csv")
sellers = pd.read_csv("../../data/processed/olist_sellers_clean.csv")
products_en = pd.read_csv(features_dir + "features_products_enriched.csv")

# Registrar
to_persist = {
    "orders": orders, "items": items, "reviews": reviews,
    "order_items": order_items, "sellers": sellers, "products_en": products_en
}
for name, df in to_persist.items():
    con.register(f"{name}_df", df)
    con.execute(f"CREATE OR REPLACE TABLE olist.{name} AS SELECT * FROM {name}_df")
    con.unregister(f"{name}_df")

In [11]:
# VISTA VENTAS
con.execute("""
CREATE OR REPLACE VIEW olist.vw_sales AS
SELECT
    i.order_id,
    i.item_count                              AS items_per_order,
    i.order_price_total                       AS price,
    i.order_freight_total                     AS freight_value,
    (i.order_price_total + i.order_freight_total) AS order_total_value,
    i.seller_count,
    o.order_year, o.order_month, o.order_dow
FROM olist.items i
LEFT JOIN olist.orders o USING(order_id);
""")

In [12]:
# VISTA LOGÍSTICA
con.execute("""
CREATE OR REPLACE VIEW olist.vw_logistics AS
SELECT
    order_id,
    delivery_days,
    estimated_days,
    delay_vs_estimated,
    late_days,
    on_time,
    prep_hours,
    transit_days,
    order_year, order_month, order_week, order_dow,
    purchase_hour, is_weekend_purchase,
    delay_bucket, delivery_days_bucket
FROM olist.orders;
""")

In [13]:
# VISTA SATISFACCION
con.execute("""
CREATE OR REPLACE VIEW olist.vw_customer_satisfaction AS
SELECT
  r.order_id,
  r.review_score,
  CAST(r.review_creation_date    AS TIMESTAMP) AS review_creation_date,
  CAST(r.review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
  CASE
    WHEN o.order_delivered_customer_date IS NULL THEN NULL
    ELSE GREATEST(
      date_diff('hour', CAST(o.order_delivered_customer_date AS TIMESTAMP),
                        CAST(r.review_creation_date          AS TIMESTAMP)), 0
    )
  END AS review_after_delivery_hours
FROM olist.reviews r
LEFT JOIN olist.orders o USING(order_id);
""")

In [14]:
# VISTA VENDEDORES
con.execute("""
CREATE OR REPLACE VIEW olist.vw_sellers AS
WITH base AS (
    SELECT
        s.seller_id,
        COUNT(DISTINCT oi.order_id)                    AS total_orders,
        COUNT(oi.order_item_id)                        AS total_items,
        SUM(oi.price + oi.freight_value)               AS total_gmv,
        AVG(o.delivery_days)                           AS avg_delivery_days,
        AVG(o.delay_vs_estimated)                      AS avg_delay,
        AVG(r.review_score)                            AS avg_review_score
    FROM olist.sellers s
    LEFT JOIN olist.order_items oi ON s.seller_id = oi.seller_id
    LEFT JOIN olist.orders o       ON oi.order_id = o.order_id
    LEFT JOIN olist.reviews r      ON oi.order_id = r.order_id
    GROUP BY s.seller_id
)
SELECT * FROM base;
""")

In [15]:
# VISTA CATEGORIAS Y MES
con.execute("""
CREATE OR REPLACE VIEW olist.vw_categories AS
WITH items_cat AS (
  SELECT
    oi.order_id, oi.product_id, oi.price, oi.freight_value,
    o.order_purchase_timestamp,
    CAST(strftime(o.order_purchase_timestamp, '%Y-%m') AS VARCHAR) AS order_month,
    EXTRACT(YEAR FROM o.order_purchase_timestamp)                  AS order_year,
    p.product_category_name_english
  FROM olist.order_items oi
  LEFT JOIN olist.orders      o ON oi.order_id  = o.order_id
  LEFT JOIN olist.products_en p ON oi.product_id = p.product_id
)
SELECT
  product_category_name_english,
  order_year,
  order_month,
  COUNT(*)                 AS category_items,
  COUNT(DISTINCT order_id) AS category_orders,
  SUM(price)               AS category_gmv,
  SUM(freight_value)       AS category_freight
FROM items_cat
GROUP BY 1,2,3
ORDER BY 2,3,1;
""")

# Resultados

In [ ]:
for v in ["vw_logistics","vw_sales","vw_customer_satisfaction","vw_sellers","vw_categories"]:
    print(f"\n--- olist.{v} ---")
    print(con.execute(f"SELECT * FROM olist.{v} LIMIT 5").fetchdf())

# 6) Exportar directo desde DuckDB
for v in ["vw_logistics","vw_sales","vw_customer_satisfaction","vw_sellers","vw_categories"]:
    con.execute(f"""
        COPY (SELECT * FROM olist.{v})
        TO '{os.path.join(views_dir, v + ".csv")}'
        WITH (HEADER, DELIMITER ',');
    """)
print("Views exported to:", views_dir)



--- olist.vw_logistics ---
                           order_id  delivery_days  estimated_days  \
0  10a045cdf6a5650c21e9cfeb60384c16            NaN       12.270625   
1  b059ee4de278302d550a3035c4cdb740            NaN       26.155532   
2  a2ac6dad85cf8af5b0afb510a240fe8c            NaN       12.211470   
3  616fa7d4871b87832197b2a137a115d2            NaN       21.354063   
4  392ed9afd714e3c74767d0c4d3e3f477            NaN       15.615937   

   delay_vs_estimated  late_days  on_time  prep_hours  transit_days  \
0                 NaN        NaN    False         NaN           NaN   
1                 NaN        NaN    False         NaN           NaN   
2                 NaN        NaN    False         NaN           NaN   
3                 NaN        NaN    False         NaN           NaN   
4                 NaN        NaN    False         NaN           NaN   

   order_year order_month  order_week  order_dow  purchase_hour  \
0        2018     2018-10          42          2         

# Guardado

In [17]:
output_dir = "../../data/views/"
os.makedirs(output_dir, exist_ok=True)

schema = "olist"
views = ["vw_logistics", "vw_sales", "vw_customer_satisfaction", "vw_sellers", "vw_categories"]

def qp(path):
    return os.path.abspath(path).replace("\\", "/")

# --- CSV ---
for v in views:
    full = f"{schema}.{v}"
    con.execute(f"""
        COPY (SELECT * FROM {full})
        TO '{qp(os.path.join(output_dir, v + ".csv"))}'
        WITH (HEADER, DELIMITER ',');
    """)

# --- Parquet  ---
for v in views:
    full = f"{schema}.{v}"
    con.execute(f"""
        COPY (SELECT * FROM {full})
        TO '{qp(os.path.join(output_dir, v + ".parquet"))}'
        (FORMAT PARQUET);
    """)

print("Views exported to:", output_dir)


Views exported to: ../../data/views/


In [18]:
con.close()